In [1]:
# 1. Install required packages
!pip install ultralytics -q

import cv2
import os
import csv
import numpy as np
from datetime import datetime
from ultralytics import YOLO

# 2. Automatically locate multiple video files in the Kaggle input directory
dataset_dir = "/kaggle/input/"
video_paths = []

for root, dirs, files in os.walk(dataset_dir):
    for file in files:
        if file.endswith(".mp4"):
            video_paths.append(os.path.join(root, file))

# Limit to a maximum of 4 videos as requested
video_paths = video_paths[:4]
print(f"✅ Found {len(video_paths)} video streams to process.")

# 3. Initialize YOLOv8 Model
model = YOLO("yolov8m.pt")

# Initialize global CSV Report File
csv_report_path = "kinematic_stress_multi_report.csv"
with open(csv_report_path, mode='w', newline='') as f:
    writer_csv = csv.writer(f)
    writer_csv.writerow(["Video_ID", "Timestamp", "Location_ID", "Detected_Condition", "Risk_Score", "Active_Persons"])

# 4. Multi-Video Processing Loop (Generates 3-4 separate output files)
for idx, video_path in enumerate(video_paths, start=1):
    output_video_path = f"output_kinematic_stress_stream_{idx}.mp4"
    print(f"\n🚀 Processing Stream {idx}/{len(video_paths)}: {os.path.basename(video_path)}")
    
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1280)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 720)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25

    writer = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    
    frame_idx = 0
    track_history = {}

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_idx += 1
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Run YOLOv8 ByteTrack tracking
        results = model.track(frame, classes=[0], persist=True, verbose=False, tracker="bytetrack.yaml")
        
        person_boxes = []
        track_ids = []
        
        if results and results[0].boxes and results[0].boxes.xyxy is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            t_ids = results[0].boxes.id
            t_ids = t_ids.cpu().numpy() if t_ids is not None else list(range(len(boxes)))
            
            for box, tid in zip(boxes, t_ids):
                x1, y1, x2, y2 = map(int, box[:4])
                person_boxes.append((x1, y1, x2, y2))
                track_ids.append(int(tid))

        active_count = len(person_boxes)
        max_risk = 0.0
        
        for box, tid in zip(person_boxes, track_ids):
            x1, y1, x2, y2 = box
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            
            if tid in track_history:
                prev_x, prev_y, _ = track_history[tid]
                displacement = np.sqrt((center_x - prev_x)**2 + (center_y - prev_y)**2)
                stress_score = min(1.0, float(displacement / 30.0))
            else:
                stress_score = 0.1
                
            track_history[tid] = (center_x, center_y, frame_idx)
            max_risk = max(max_risk, stress_score)
            
            if stress_score > 0.65:
                color = (0, 0, 255)
                status_text = f"ID: {tid} | HIGH STRUGGLE ({stress_score:.2f})"
            elif stress_score > 0.35:
                color = (0, 165, 255)
                status_text = f"ID: {tid} | STRAIN ({stress_score:.2f})"
            else:
                color = (0, 255, 120)
                status_text = f"ID: {tid} | STABLE ({stress_score:.2f})"
                
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, status_text, (x1, max(25, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        condition_status = "CRITICAL STRUGGLE ALERT" if max_risk > 0.65 else "NORMAL SURVEILLANCE FLOW"
        banner_color = (0, 0, 255) if max_risk > 0.65 else (0, 140, 255)
        
        cv2.rectangle(frame, (0, 0), (width, 65), (15, 15, 15), -1)
        cv2.putText(frame, f"STREAM {idx} | ACTIVE PERSONS: {active_count}", (20, 28), cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
        cv2.putText(frame, f"STATUS: {condition_status} | PEAK RISK: {max_risk:.2f}", (20, 52), cv2.FONT_HERSHEY_DUPLEX, 0.6, banner_color, 2)

        if frame_idx % 25 == 0:
            with open(csv_report_path, mode='a', newline='') as f:
                writer_csv = csv.writer(f)
                writer_csv.writerow([f"Stream_{idx}", timestamp, f"Zone-{idx}", condition_status, round(max_risk, 2), active_count])

        writer.write(frame)

    cap.release()
    writer.release()
    print(f"✅ Saved: '{output_video_path}'")

print(f"\n🎉 All processing complete! Generated {len(video_paths)} separate output videos and consolidated report '{csv_report_path}'.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 2.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
✅ Found 4 video streams to process.

🚀 Processing Stream 1/4: V_465.mp4
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 400ms
 Downloaded lap
Prepared 1 package in 85ms
Installed 1 package in 10ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart r

In [2]:
from IPython.display import HTML, display

html_code = """
<div style="display: flex; flex-wrap: wrap; gap: 10px; justify-content: center;">
    <div style="width: 48%;">
        <p><b>Stream 1 (V_465.mp4)</b></p>
        <video width="100%" controls>
            <source src="output_kinematic_stress_stream_1.mp4" type="video/mp4">
        </video>
    </div>
    <div style="width: 48%;">
        <p><b>Stream 2 (V_313.mp4)</b></p>
        <video width="100%" controls>
            <source src="output_kinematic_stress_stream_2.mp4" type="video/mp4">
        </video>
    </div>
    <div style="width: 48%;">
        <p><b>Stream 3 (V_753.mp4)</b></p>
        <video width="100%" controls>
            <source src="output_kinematic_stress_stream_3.mp4" type="video/mp4">
        </video>
    </div>
    <div style="width: 48%;">
        <p><b>Stream 4 (V_963.mp4)</b></p>
        <video width="100%" controls>
            <source src="output_kinematic_stress_stream_4.mp4" type="video/mp4">
        </video>
    </div>
</div>
"""

display(HTML(html_code))

In [3]:
# Convert output videos to web-friendly format using ffmpeg
for i in range(1, 5):
    !ffmpeg -y -i output_kinematic_stress_stream_{i}.mp4 -vcodec libx264 -acodec aac web_stream_{i}.mp4 -q:v 5 > /dev/null 2>&1

print("✅ Conversion complete! Run the display cell below.")

✅ Conversion complete! Run the display cell below.


In [4]:
from IPython.display import HTML, display

html_code = """
<div style="display: flex; flex-wrap: wrap; gap: 10px; justify-content: center;">
    <div style="width: 48%;">
        <p><b>Stream 1 (V_465.mp4)</b></p>
        <video width="100%" controls>
            <source src="web_stream_1.mp4" type="video/mp4">
        </video>
    </div>
    <div style="width: 48%;">
        <p><b>Stream 2 (V_313.mp4)</b></p>
        <video width="100%" controls>
            <source src="web_stream_2.mp4" type="video/mp4">
        </video>
    </div>
    <div style="width: 48%;">
        <p><b>Stream 3 (V_753.mp4)</b></p>
        <video width="100%" controls>
            <source src="web_stream_3.mp4" type="video/mp4">
        </video>
    </div>
    <div style="width: 48%;">
        <p><b>Stream 4 (V_963.mp4)</b></p>
        <video width="100%" controls>
            <source src="web_stream_4.mp4" type="video/mp4">
        </video>
    </div>
</div>
"""

display(HTML(html_code))

In [5]:
import random
import os

dataset_dir = "/kaggle/input/"
all_videos = []

for root, dirs, files in os.walk(dataset_dir):
    for file in files:
        if file.endswith(".mp4"):
            all_videos.append(os.path.join(root, file))

# Randomly select 4 distinct videos
new_video_paths = random.sample(all_videos, min(4, len(all_videos)))

print(f"✅ Randomly selected {len(new_video_paths)} new videos:")
for path in new_video_paths:
    print(f" - {os.path.basename(path)}")

✅ Randomly selected 4 new videos:
 - NV_202.mp4
 - V_209.mp4
 - V_716.mp4
 - NV_789.mp4


In [6]:
# 1. Run the processing loop on your new batch
for idx, video_path in enumerate(new_video_paths, start=1):
    output_video_path = f"output_kinematic_stress_stream_{idx}.mp4"
    print(f"\n🚀 Processing New Stream {idx}: {os.path.basename(video_path)}")
    
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1280)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 720)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25

    writer = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    
    frame_idx = 0
    track_history = {}

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_idx += 1
        results = model.track(frame, classes=[0], persist=True, verbose=False, tracker="bytetrack.yaml")
        
        person_boxes = []
        track_ids = []
        
        if results and results[0].boxes and results[0].boxes.xyxy is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            t_ids = results[0].boxes.id
            t_ids = t_ids.cpu().numpy() if t_ids is not None else list(range(len(boxes)))
            
            for box, tid in zip(boxes, t_ids):
                x1, y1, x2, y2 = map(int, box[:4])
                person_boxes.append((x1, y1, x2, y2))
                track_ids.append(int(tid))

        active_count = len(person_boxes)
        max_risk = 0.0
        
        for box, tid in zip(person_boxes, track_ids):
            x1, y1, x2, y2 = box
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            
            if tid in track_history:
                prev_x, prev_y, _ = track_history[tid]
                displacement = np.sqrt((center_x - prev_x)**2 + (center_y - prev_y)**2)
                stress_score = min(1.0, float(displacement / 30.0))
            else:
                stress_score = 0.1
                
            track_history[tid] = (center_x, center_y, frame_idx)
            max_risk = max(max_risk, stress_score)
            
            if stress_score > 0.65:
                color = (0, 0, 255)
                status_text = f"ID: {tid} | HIGH STRUGGLE ({stress_score:.2f})"
            elif stress_score > 0.35:
                color = (0, 165, 255)
                status_text = f"ID: {tid} | STRAIN ({stress_score:.2f})"
            else:
                color = (0, 255, 120)
                status_text = f"ID: {tid} | STABLE ({stress_score:.2f})"
                
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, status_text, (x1, max(25, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        condition_status = "CRITICAL STRUGGLE ALERT" if max_risk > 0.65 else "NORMAL SURVEILLANCE FLOW"
        banner_color = (0, 0, 255) if max_risk > 0.65 else (0, 140, 255)
        
        cv2.rectangle(frame, (0, 0), (width, 65), (15, 15, 15), -1)
        cv2.putText(frame, f"NEW STREAM {idx} | ACTIVE PERSONS: {active_count}", (20, 28), cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
        cv2.putText(frame, f"STATUS: {condition_status} | PEAK RISK: {max_risk:.2f}", (20, 52), cv2.FONT_HERSHEY_DUPLEX, 0.6, banner_color, 2)

        writer.write(frame)

    cap.release()
    writer.release()

# 2. Re-run ffmpeg conversion for the new batch
for i in range(1, 5):
    !ffmpeg -y -i output_kinematic_stress_stream_{i}.mp4 -vcodec libx264 -acodec aac web_stream_{i}.mp4 -q:v 5 > /dev/null 2>&1

print("✅ New batch processed, converted, and ready to display!")


🚀 Processing New Stream 1: NV_202.mp4

🚀 Processing New Stream 2: V_209.mp4

🚀 Processing New Stream 3: V_716.mp4

🚀 Processing New Stream 4: NV_789.mp4
✅ New batch processed, converted, and ready to display!


In [7]:
from IPython.display import HTML, display

html_code = """
<div style="display: flex; flex-wrap: wrap; gap: 10px; justify-content: center;">
    <div style="width: 48%;">
        <p><b>New Stream 1 (NV_70.mp4)</b></p>
        <video width="100%" controls>
            <source src="web_stream_1.mp4" type="video/mp4">
        </video>
    </div>
    <div style="width: 48%;">
        <p><b>New Stream 2 (V_781.mp4)</b></p>
        <video width="100%" controls>
            <source src="web_stream_2.mp4" type="video/mp4">
        </video>
    </div>
    <div style="width: 48%;">
        <p><b>New Stream 3 (V_555.mp4)</b></p>
        <video width="100%" controls>
            <source src="web_stream_3.mp4" type="video/mp4">
        </video>
    </div>
    <div style="width: 48%;">
        <p><b>New Stream 4 (NV_735.mp4)</b></p>
        <video width="100%" controls>
            <source src="web_stream_4.mp4" type="video/mp4">
        </video>
    </div>
</div>
"""

display(HTML(html_code))